# DSA 210 - Phase 4: Machine Learning Methods
## Music Listening Behavior & Weather Analysis
**Deha Ozan | Spring 2026**

## 0. Setup & Data Loading

In [ ]:
# Install / import everything needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)
print('Libraries loaded.')

In [ ]:
# Load data directly from GitHub — no upload needed
URL = 'https://raw.githubusercontent.com/deha-ozan/DSA210projectDeha/main/music_weather_daily.csv'
daily = pd.read_csv(URL, parse_dates=['date'])

print(f'Shape: {daily.shape}')
print(f'Date range: {daily["date"].min().date()} to {daily["date"].max().date()}')
daily.head()

---
## 1. Linear Regression

**Question:** Can daily weather conditions linearly predict how many minutes I listen to music?

**Features:** `temp_mean`, `temp_max`, `precipitation`, `sunshine_hours` | **Target:** `total_listen_min`

In [ ]:
X_lr = daily[['temp_mean', 'temp_max', 'precipitation', 'sunshine_hours']]
y_lr = daily['total_listen_min']

X_train, X_test, y_train, y_test = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

lr_r2  = r2_score(y_test, y_pred)
lr_mae = mean_absolute_error(y_test, y_pred)

print(f'R2 : {lr_r2:.4f}')
print(f'MAE: {lr_mae:.2f} minutes')
print('\nCoefficients:')
for f, c in zip(X_lr.columns, lr.coef_):
    print(f'  {f}: {c:+.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Linear Regression: Weather to Listening Duration', fontweight='bold')

axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.4)
axes[0].plot([0,600],[0,600],'r--', label='Perfect fit')
axes[0].set_xlabel('Actual (min)'); axes[0].set_ylabel('Predicted (min)')
axes[0].set_title(f'Actual vs Predicted  (R2={lr_r2:.3f})')
axes[0].legend()

axes[1].scatter(y_pred, y_test - y_pred, alpha=0.5, color='coral', edgecolors='white', linewidth=0.4)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted (min)'); axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.tight_layout(); plt.show()

# Coefficient chart
coefs = pd.Series(lr.coef_, index=X_lr.columns)
colors = ['green' if c > 0 else 'tomato' for c in coefs]
coefs.plot(kind='barh', color=colors, edgecolor='white', figsize=(7,3))
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Coefficients'); plt.tight_layout(); plt.show()

**Interpretation:** R2 = 0.05 — weather features explain almost none of the variance in listening duration. The residuals show no pattern, meaning the model is not misspecified — there are simply important variables missing. This motivates more flexible, feature-rich methods.

---
## 2. Logistic Regression

**Question:** Does listening behavior predict whether it was a rainy day?

**Features:** `n_plays`, `total_listen_min`, `skip_rate`, `fraction_complete` | **Target:** `is_rainy`

In [ ]:
X_log = daily[['n_plays', 'total_listen_min', 'skip_rate', 'fraction_complete']]
y_log = daily['is_rainy'].astype(int)

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_log, y_log, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train2s = scaler.fit_transform(X_train2)
X_test2s  = scaler.transform(X_test2)

logr = LogisticRegression(max_iter=1000, random_state=42)
logr.fit(X_train2s, y_train2)
y_pred2 = logr.predict(X_test2s)

log_acc  = accuracy_score(y_test2, y_pred2)
baseline = y_test2.value_counts(normalize=True).max()

print(f'Accuracy:          {log_acc:.4f}')
print(f'Majority baseline: {baseline:.4f}')
print(f'Lift over baseline:{log_acc - baseline:+.4f}')
print()
print(classification_report(y_test2, y_pred2, target_names=['Not Rainy','Rainy'], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Logistic Regression: Listening Behavior to Rainy Day', fontweight='bold')

cm = confusion_matrix(y_test2, y_pred2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Rainy','Rainy'], yticklabels=['Not Rainy','Rainy'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

bars = axes[1].bar(['Majority Baseline','Logistic Regression'],
                   [baseline, log_acc], color=['tomato','steelblue'], width=0.5, edgecolor='white')
axes[1].set_ylim(0,1); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model vs Baseline')
for bar, val in zip(bars, [baseline, log_acc]):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout(); plt.show()

**Interpretation:** Accuracy = 70.6%, identical to the majority-class baseline. The model predicts 'not rainy' almost every time — listening patterns carry no discriminative signal about rain. This is consistent with Phase 3 results and confirms the relationship is not symmetric.

---
## 3. K-Nearest Neighbors Regression

**Question:** Can a nonlinear model using all features do better at predicting listening time?

**Features:** `temp_mean`, `precipitation`, `sunshine_hours`, `n_plays`, `skip_rate`, `day_of_week`, `month` | **Target:** `total_listen_min`

In [ ]:
X_knn = daily[['temp_mean','precipitation','sunshine_hours','n_plays','skip_rate','day_of_week','month']]
y_knn = daily['total_listen_min']

X_train3, X_test3, y_train3, y_test3 = train_test_split(X_knn, y_knn, test_size=0.2, random_state=42)
sc3 = StandardScaler()
X_train3s = sc3.fit_transform(X_train3)
X_test3s  = sc3.transform(X_test3)

knn_results = []
for k in [1, 3, 5, 10, 20, 50]:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train3s, y_train3)
    p = knn.predict(X_test3s)
    knn_results.append({'k': k, 'R2': r2_score(y_test3, p), 'MAE': mean_absolute_error(y_test3, p)})

knn_df = pd.DataFrame(knn_results)
print(knn_df.to_string(index=False))
best_k = int(knn_df.loc[knn_df['R2'].idxmax(), 'k'])
print(f'\nBest k: {best_k}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(knn_df['k'], knn_df['R2'], 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].set_xlabel('k'); axes[0].set_ylabel('R2'); axes[0].set_title('R2 by k'); axes[0].grid(alpha=0.3)
axes[1].plot(knn_df['k'], knn_df['MAE'], 'o-', color='coral', linewidth=2, markersize=7)
axes[1].set_xlabel('k'); axes[1].set_ylabel('MAE (min)'); axes[1].set_title('MAE by k'); axes[1].grid(alpha=0.3)
plt.suptitle('KNN: k Tuning', fontweight='bold'); plt.tight_layout(); plt.show()

knn_best = KNeighborsRegressor(n_neighbors=best_k)
knn_best.fit(X_train3s, y_train3)
y_pred3 = knn_best.predict(X_test3s)
knn_r2  = r2_score(y_test3, y_pred3)
knn_mae = mean_absolute_error(y_test3, y_pred3)
print(f'Best KNN (k={best_k}): R2={knn_r2:.4f}, MAE={knn_mae:.2f} min')

plt.figure(figsize=(6,5))
plt.scatter(y_test3, y_pred3, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.4)
plt.plot([0,600],[0,600],'r--', label='Perfect fit')
plt.xlabel('Actual (min)'); plt.ylabel('Predicted (min)')
plt.title(f'KNN (k={best_k}): Actual vs Predicted  (R2={knn_r2:.3f})')
plt.legend(); plt.show()

**Interpretation:** Best KNN (k=10) achieves R2=0.77, MAE=27 min — a huge jump from linear regression (R2=0.05). Adding time and behavior features alongside weather, with a nonlinear method, reveals complex patterns. k=1 overfits; k=50 over-smooths. k=10 is the sweet spot.

---
## 4. Decision Tree Classification

**Question:** Can listening behavior and time features predict the weather condition?

**Features:** `n_plays`, `total_listen_min`, `skip_rate`, `fraction_complete`, `day_of_week`, `month`, `year` *(no weather features)*  | **Target:** `condition`

In [ ]:
le = LabelEncoder()
y_dt = le.fit_transform(daily['condition'])
X_dt = daily[['n_plays','total_listen_min','skip_rate','fraction_complete','day_of_week','month','year']]

X_train4, X_test4, y_train4, y_test4 = train_test_split(X_dt, y_dt, test_size=0.2, random_state=42)

dt_results = []
for d in [2, 3, 4, 5, 7, 10, None]:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train4, y_train4)
    dt_results.append({'max_depth': str(d), 'Accuracy': accuracy_score(y_test4, dt.predict(X_test4))})

dt_df = pd.DataFrame(dt_results)
print(dt_df.to_string(index=False))
print(f'Majority baseline: {pd.Series(y_test4).value_counts(normalize=True).max():.4f}')

In [ ]:
dt_best = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_best.fit(X_train4, y_train4)
y_pred4 = dt_best.predict(X_test4)
dt_acc  = accuracy_score(y_test4, y_pred4)

print(f'Decision Tree (depth=3): Accuracy={dt_acc:.4f}')
print(classification_report(y_test4, y_pred4, target_names=le.classes_, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Decision Tree: Listening+Time to Weather Condition', fontweight='bold')

cm_dt = confusion_matrix(y_test4, y_pred4)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[0].set_title('Confusion Matrix')

fi = pd.Series(dt_best.feature_importances_, index=X_dt.columns).sort_values()
fi.plot(kind='barh', ax=axes[1], edgecolor='white', color='steelblue')
axes[1].set_title('Feature Importances')

plt.tight_layout(); plt.show()

# Tree structure
fig, ax = plt.subplots(figsize=(16,6))
plot_tree(dt_best, feature_names=X_dt.columns, class_names=le.classes_,
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.title('Decision Tree Structure (max_depth=3)'); plt.show()

**Interpretation:** Accuracy = 49% on a 4-class problem (baseline ~50%). Month dominates feature importance (53%) — weather condition is mostly encoded by time of year, not listening patterns. Listening features add minimal discriminative power, confirming the asymmetry found across all methods.

---
## 5. Summary

In [ ]:
print('='*55)
print('PHASE 4 RESULTS SUMMARY')
print('='*55)
print(f'Linear Regression    R2={lr_r2:.3f}  MAE={lr_mae:.1f} min')
print(f'Logistic Regression  Accuracy={log_acc:.3f}  (baseline={baseline:.3f})')
print(f'KNN (k={best_k})          R2={knn_r2:.3f}  MAE={knn_mae:.1f} min')
print(f'Decision Tree (d=3)  Accuracy={dt_acc:.3f}')
print()
print('Key finding: Weather alone is a weak linear predictor (R2=0.05).')
print('Combining all features nonlinearly (KNN) jumps to R2=0.77.')
print('Listening behavior cannot predict rain in either direction.')

### Conclusions

| Finding | Evidence |
|---------|----------|
| Weather has weak linear predictive power over listening time | Linear R2 = 0.05 |
| Listening behavior cannot predict rain | Logistic accuracy = baseline (71%) |
| Nonlinear model with all features dramatically improves prediction | KNN R2 = 0.77 |
| Weather condition is encoded in time features, not listening patterns | DT: month importance = 53% |

Listening duration is predictable — but from *when* in the year and weekly routines, not weather itself. The relationship between weather and music listening is weaker and more complex than the initial hypothesis suggested.